# ArSL MediaPipe (Keypoints) — Kaggle Edition

Arabic Sign Language letter recognition | MediaPipe Hands → 63 keypoints/features → MLP classifier

**Pipeline:** auto-detect dataset | optional keypoint extraction | tf.data + AUTOTUNE | two-phase training | confusion matrix

## 1. Setup & Imports

In [ ]:
import os
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix

print(f'TensorFlow: {tf.__version__}')


In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────
BATCH_SIZE      = 256   # keypoints are small; large batch is usually fine
INITIAL_EPOCHS  = 30
FINETUNE_EPOCHS = 30
SEED            = 42
VAL_SPLIT       = 0.2
TEST_SPLIT      = 0.2

# Extraction controls (only used if we need to generate keypoints from images)
EXTRACT_LIMIT = None  # e.g. 2000 for a quick test; None = full dataset
MIN_DET_CONF  = 0.5

OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/input') else '.'
tf.keras.utils.set_random_seed(SEED)

print(f'OUTPUT_DIR={OUTPUT_DIR} | BATCH_SIZE={BATCH_SIZE} | SEED={SEED}')


In [ ]:
# ── GPU / Performance setup ───────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Mixed precision can speed up on modern GPUs; keep output float32.
USE_MIXED_PRECISION = bool(gpus)
if USE_MIXED_PRECISION:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print('Mixed precision enabled: mixed_float16')
    except Exception as e:
        print(f'Mixed precision not enabled: {e}')
        tf.keras.mixed_precision.set_global_policy('float32')
else:
    tf.keras.mixed_precision.set_global_policy('float32')

print('GPUs:', [g.name for g in gpus] if gpus else ['None - CPU'])
AUTOTUNE = tf.data.AUTOTUNE


## 2. Dataset Auto-Detection

This notebook supports:
- **Keypoints CSV** (preferred): a CSV containing 63 keypoint features + a label column
- **Image labels CSV**: a CSV containing image paths + labels (we extract keypoints with MediaPipe)
- **Folder per class**: `train/<class>/*.jpg` (we extract keypoints with MediaPipe)

In [ ]:
DATASET_MODE   = None  # 'keypoints_csv' | 'image_csv' | 'folder_train'
DATA_ROOT      = None
TRAIN_DIR      = None
LABELS_CSV     = None
KEYPOINTS_CSV  = None

def _walk_limited(root, max_depth=5):
    for r, d, f in os.walk(root):
        level = r.replace(root, '').count(os.sep)
        if level > max_depth:
            d[:] = []
            continue
        yield r, d, f

def _pick_col(columns, candidates):
    cols = {c.lower(): c for c in columns}
    for c in candidates:
        if c in cols:
            return cols[c]
    return None

def _score_keypoints_csv(path):
    name = os.path.basename(path).lower()
    score = 0
    if 'keypoint' in name or 'mediapipe' in name:
        score += 5
    try:
        head = pd.read_csv(path, nrows=5)
    except Exception:
        return -1
    label_col = _pick_col(head.columns, ['label', 'letter', 'class', 'category', 'y'])
    if label_col is None:
        return -1
    # heuristic: at least 63 numeric-ish feature columns
    feature_cols = [c for c in head.columns if c != label_col]
    if len(feature_cols) >= 63:
        score += 10
    # landmark-style columns x0..z20
    if all(any(f'{axis}{i}' in (c.lower()) for c in feature_cols) for axis in ['x', 'y', 'z'] for i in [0, 10, 20]):
        score += 5
    return score

def _score_image_labels_csv(path):
    name = os.path.basename(path).lower()
    score = 0
    if 'label' in name or 'train' in name or 'annot' in name:
        score += 3
    try:
        head = pd.read_csv(path, nrows=5)
    except Exception:
        return -1
    label_col = _pick_col(head.columns, ['label', 'letter', 'class', 'category', 'y'])
    path_col  = _pick_col(head.columns, ['path', 'filepath', 'file_path', 'image', 'filename', 'file'])
    return (score + 10) if (label_col and path_col) else -1

def _find_best_csv(root, scorer):
    cand = []
    for r, _, files in _walk_limited(root, max_depth=6):
        for fn in files:
            if not fn.lower().endswith('.csv'):
                continue
            p = os.path.join(r, fn)
            s = scorer(p)
            if s >= 0:
                cand.append((s, p))
    if not cand:
        return None
    cand.sort(key=lambda x: (x[0], -len(x[1])), reverse=True)
    return cand[0][1]

def _find_train_dir(root):
    for r, d, _ in _walk_limited(root, max_depth=6):
        if 'train' in d:
            return os.path.join(r, 'train')
    return None

if os.path.exists('/kaggle/input'):
    DATA_ROOT = '/kaggle/input'
    TRAIN_DIR = _find_train_dir(DATA_ROOT)
    KEYPOINTS_CSV = _find_best_csv(DATA_ROOT, _score_keypoints_csv)
    if KEYPOINTS_CSV:
        DATASET_MODE = 'keypoints_csv'
        print('Mode: KEYPOINTS_CSV')
        print('KEYPOINTS_CSV:', KEYPOINTS_CSV)
    else:
        LABELS_CSV = _find_best_csv(DATA_ROOT, _score_image_labels_csv)
        if LABELS_CSV and TRAIN_DIR:
            DATASET_MODE = 'image_csv'
            print('Mode: IMAGE_CSV (train/ + labels.csv)')
            print('TRAIN_DIR :', TRAIN_DIR)
            print('LABELS_CSV:', LABELS_CSV)
        elif TRAIN_DIR:
            DATASET_MODE = 'folder_train'
            print('Mode: FOLDER_TRAIN (train/<class>/...)')
            print('TRAIN_DIR :', TRAIN_DIR)
        else:
            raise FileNotFoundError('No compatible dataset found under /kaggle/input')
else:
    # Local fallback: point these manually if needed
    DATA_ROOT = '.'
    print('Local mode: set KEYPOINTS_CSV or (TRAIN_DIR + LABELS_CSV) manually if auto-detect fails.')

print('DATASET_MODE:', DATASET_MODE)


## 3. Load / Extract Keypoints Dataset

If a keypoints CSV is found, it will be used directly. Otherwise, the notebook will extract keypoints from images using MediaPipe Hands.

In [ ]:
def _resolve_paths(paths, root_hint, train_dir=None):
    resolved = []
    for p in paths:
        p2 = str(p).replace('\\', '/')
        if os.path.isabs(p2) and os.path.exists(p2):
            resolved.append(p2)
            continue
        cand1 = os.path.join(root_hint, p2)
        cand2 = os.path.join(train_dir or root_hint, p2)
        if os.path.exists(cand1):
            resolved.append(cand1)
        elif os.path.exists(cand2):
            resolved.append(cand2)
        else:
            resolved.append(os.path.join(train_dir or root_hint, os.path.basename(p2)))
    return resolved


def _load_keypoints_csv(path):
    df = pd.read_csv(path)
    label_col = None
    for c in ['label', 'letter', 'class', 'category', 'y']:
        if c in [x.lower() for x in df.columns]:
            label_col = next(cc for cc in df.columns if cc.lower() == c)
            break
    if label_col is None:
        raise KeyError('No label column found in keypoints CSV.')
    feature_cols = [c for c in df.columns if c != label_col]
    if len(feature_cols) < 63:
        raise ValueError(f'Expected >=63 feature columns, got {len(feature_cols)}')
    X = df[feature_cols].astype('float32').values
    y = df[label_col].astype(str).str.strip().values
    return X, y, feature_cols, label_col


X = y = None
feature_cols = None
label_encoder = LabelEncoder()
class_names = None
num_classes = None

if DATASET_MODE == 'keypoints_csv':
    X, y, feature_cols, label_col = _load_keypoints_csv(KEYPOINTS_CSV)
    print('Loaded keypoints CSV:', KEYPOINTS_CSV)
    print('X shape:', X.shape)

else:
    # We need MediaPipe only for extraction
    try:
        import mediapipe as mp
    except Exception as e:
        raise ImportError(
            'MediaPipe is required to extract keypoints from images but is not available.\n'
            'Install mediapipe (Kaggle/local) or provide a keypoints CSV instead.\n'
            f'Original error: {e}'
        )

    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=MIN_DET_CONF,
    )

    rows = []
    labels = []

    if DATASET_MODE == 'image_csv':
        df = pd.read_csv(LABELS_CSV)
        path_col  = _pick_col(df.columns, ['path', 'filepath', 'file_path', 'image', 'filename', 'file'])
        label_col = _pick_col(df.columns, ['label', 'letter', 'class', 'category', 'y'])
        if path_col is None or label_col is None:
            raise KeyError('labels CSV must include a path column and a label column.')
        df = df[[path_col, label_col]].copy()
        df[label_col] = df[label_col].astype(str).str.strip()
        paths = _resolve_paths(df[path_col].tolist(), root_hint=DATA_ROOT, train_dir=TRAIN_DIR)
        labels_in = df[label_col].tolist()
        pairs = list(zip(paths, labels_in))
    elif DATASET_MODE == 'folder_train':
        pairs = []
        class_dirs = [d for d in sorted(os.listdir(TRAIN_DIR)) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
        for cls in class_dirs:
            cls_dir = os.path.join(TRAIN_DIR, cls)
            for fn in os.listdir(cls_dir):
                if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                    pairs.append((os.path.join(cls_dir, fn), cls))
    else:
        raise ValueError(f'Unsupported DATASET_MODE for extraction: {DATASET_MODE}')

    if EXTRACT_LIMIT is not None:
        pairs = pairs[: int(EXTRACT_LIMIT)]
    print('Extracting keypoints from images:', len(pairs))

    import cv2
    for idx, (img_path, lbl) in enumerate(pairs, 1):
        img = cv2.imread(img_path)
        if img is None:
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        res = hands.process(rgb)
        if not res.multi_hand_landmarks:
            continue
        hand = res.multi_hand_landmarks[0]
        feat = []
        for lm in hand.landmark:
            feat.extend([lm.x, lm.y, lm.z])
        if len(feat) != 63:
            continue
        rows.append(feat)
        labels.append(str(lbl))
        if idx % 2000 == 0:
            print(f'  processed {idx}/{len(pairs)} | kept {len(rows)}')

    if not rows:
        raise RuntimeError('No keypoints extracted. Check dataset paths and MediaPipe detection quality.')

    X = np.array(rows, dtype=np.float32)
    y = np.array(labels, dtype=object)

    out_csv = Path(OUTPUT_DIR) / 'arsl_mediapipe_keypoints_generated.csv'
    cols = []
    for i in range(21):
        cols.extend([f'x{i}', f'y{i}', f'z{i}'])
    df_out = pd.DataFrame(X, columns=cols)
    df_out['label'] = y
    df_out.to_csv(out_csv, index=False)
    print('Saved generated keypoints CSV:', out_csv)

# Encode labels (always, regardless of mode)
y_encoded = label_encoder.fit_transform(y)
class_names = list(label_encoder.classes_)
num_classes = len(class_names)
print('Classes:', num_classes)
print('X shape:', X.shape)


## 4. Split Data (Train / Val / Test)

In [ ]:
# Train/Test split first
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y_encoded, test_size=TEST_SPLIT, random_state=SEED, stratify=y_encoded
)

# Then split remaining into Train/Val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=VAL_SPLIT, random_state=SEED, stratify=y_train_full
)

y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   num_classes=num_classes)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  num_classes=num_classes)

print('Train:', X_train.shape, y_train_oh.shape)
print('Val  :', X_val.shape,   y_val_oh.shape)
print('Test :', X_test.shape,  y_test_oh.shape)


## 5. tf.data Pipeline

In [ ]:
SHUFFLE_BUFFER = 20000

def augment_keypoints(x, y):
    # Small gaussian noise works as keypoint jitter augmentation
    noise = tf.random.normal(tf.shape(x), mean=0.0, stddev=0.01, dtype=x.dtype)
    return x + noise, y

def make_ds(features, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((features, labels))
    if training:
        ds = ds.shuffle(min(SHUFFLE_BUFFER, len(features)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(augment_keypoints, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(X_train.astype('float32'), y_train_oh.astype('float32'), training=True)
val_ds   = make_ds(X_val.astype('float32'),   y_val_oh.astype('float32'),   training=False)
test_ds  = make_ds(X_test.astype('float32'),  y_test_oh.astype('float32'),  training=False)

print('Datasets ready.')


## 6. Model Architecture (MLP)

In [ ]:
tf.keras.backend.clear_session()

model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(512, activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    layers.Dense(256, activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    layers.Dense(128, activation='relu', kernel_initializer='he_normal'),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax', dtype='float32'),
])

model.summary()


## 7. Phase 1 — Initial Training

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

cb1 = [
    callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'mediapipe_mlp_best_initial.h5'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, verbose=1
    ),
    callbacks.CSVLogger(os.path.join(OUTPUT_DIR, 'training_initial.csv')),
]

print('=' * 60)
print('PHASE 1: INITIAL TRAINING')
print('=' * 60)
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=cb1,
)


## 8. Phase 2 — Fine-Tuning

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

cb2 = [
    callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'mediapipe_mlp_best_finetuned.h5'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, verbose=1
    ),
    callbacks.CSVLogger(os.path.join(OUTPUT_DIR, 'training_finetune.csv')),
]

print('=' * 60)
print('PHASE 2: FINE-TUNING')
print('=' * 60)
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=cb2,
)

model.save(os.path.join(OUTPUT_DIR, 'mediapipe_mlp_final.h5'))
print('Final model saved:', os.path.join(OUTPUT_DIR, 'mediapipe_mlp_final.h5'))


## 9. Training History (from CSV logs)

In [ ]:
initial_csv  = Path(OUTPUT_DIR) / 'training_initial.csv'
finetune_csv = Path(OUTPUT_DIR) / 'training_finetune.csv'

dfs = []
split_epoch = None
if initial_csv.exists():
    df1 = pd.read_csv(initial_csv)
    df1['phase'] = 'initial'
    dfs.append(df1)
    split_epoch = len(df1)
if finetune_csv.exists():
    df2 = pd.read_csv(finetune_csv)
    df2['phase'] = 'finetune'
    dfs.append(df2)

if not dfs:
    print('No CSV logs found yet. Run training cells first.')
else:
    df = pd.concat(dfs, ignore_index=True)
    # Continuous epoch index
    epoch_idx = []
    offset = 0
    for dfi in dfs:
        n = len(dfi)
        epoch_idx.extend(list(range(offset + 1, offset + n + 1)))
        offset += n
    df = df.copy()
    df['epoch_idx'] = epoch_idx

    acc_col = 'accuracy' if 'accuracy' in df.columns else None
    val_acc_col = 'val_accuracy' if 'val_accuracy' in df.columns else None
    if acc_col and val_acc_col:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(df['epoch_idx'], df[acc_col], 'o-', label='Train Acc')
        ax1.plot(df['epoch_idx'], df[val_acc_col], 'o-', label='Val Acc')
        if split_epoch and finetune_csv.exists():
            ax1.axvline(x=split_epoch + 0.5, color='green', linestyle='--', label='Fine-tune')
        ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True, alpha=0.3)

        ax2.plot(df['epoch_idx'], df['loss'], 'o-', label='Train Loss')
        ax2.plot(df['epoch_idx'], df['val_loss'], 'o-', label='Val Loss')
        if split_epoch and finetune_csv.exists():
            ax2.axvline(x=split_epoch + 0.5, color='green', linestyle='--', label='Fine-tune')
        ax2.set_title('Loss'); ax2.legend(); ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        out_path = Path(OUTPUT_DIR) / 'training_history.png'
        plt.savefig(out_path, dpi=150)
        plt.show()
        print('Saved:', out_path)
        print('Best Val Accuracy:', float(df[val_acc_col].max()))
    else:
        print('CSV columns not as expected:', df.columns.tolist())


## 10. Evaluation (Test Set)

Optional TTA-style evaluation: add small jitter noise to keypoints multiple times and average predictions.

In [ ]:
USE_TTA = False
TTA_ROUNDS = 5
TTA_NOISE_STD = 0.01

best_path = os.path.join(OUTPUT_DIR, 'mediapipe_mlp_best_finetuned.h5')
best_model = tf.keras.models.load_model(best_path) if os.path.exists(best_path) else model

if not USE_TTA:
    probs = best_model.predict(test_ds, verbose=0)
else:
    # Manual TTA over numpy test arrays
    probs_accum = 0
    for _ in range(TTA_ROUNDS):
        jitter = np.random.normal(0.0, TTA_NOISE_STD, size=X_test.shape).astype('float32')
        ds = make_ds((X_test + jitter).astype('float32'), y_test_oh.astype('float32'), training=False)
        probs_accum += best_model.predict(ds, verbose=0)
    probs = probs_accum / float(TTA_ROUNDS)

pred_classes = np.argmax(probs, axis=1)
true_classes = y_test.astype(int)
accuracy = float((pred_classes == true_classes).mean() * 100.0)

print(f'Test Accuracy: {accuracy:.2f}%')


## 11. Confusion Matrix

In [ ]:
label_set = list(range(num_classes))
cm = confusion_matrix(true_classes, pred_classes, labels=label_set)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — Test Set', fontweight='bold')
plt.tight_layout()
out_path = Path(OUTPUT_DIR) / 'confusion_matrix.png'
plt.savefig(out_path, dpi=150)
plt.show()
print('Saved:', out_path)


## 12. Final Summary

In [ ]:
print('=' * 60)
print('ARSL MEDIAPIPE (KEYPOINTS) MLP - SUMMARY')
print('=' * 60)
print('Mode:            ', DATASET_MODE)
print('Classes:         ', num_classes)
print('Batch size:      ', BATCH_SIZE)
print('Initial epochs:  ', INITIAL_EPOCHS)
print('Finetune epochs: ', FINETUNE_EPOCHS)
print(f'Test Accuracy:    {accuracy:.2f}%')
print('Mixed precision: ', 'ON' if USE_MIXED_PRECISION else 'OFF')
print()
for fname in [
    'mediapipe_mlp_best_initial.h5',
    'mediapipe_mlp_best_finetuned.h5',
    'mediapipe_mlp_final.h5',
    'training_initial.csv',
    'training_finetune.csv',
    'training_history.png',
    'confusion_matrix.png',
    'arsl_mediapipe_keypoints_generated.csv',
]:
    p = Path(OUTPUT_DIR) / fname
    mark = 'OK' if p.exists() else '--'
    print(f'  [{mark}] {fname}')
